# VAE-DiT TTS — Inference Demo

Multilingual zero-shot TTS with voice cloning. Supports Chinese, Japanese, and English.

**Features:**
- 🎙️ Clone any voice with 3-10s reference audio
- 🌏 Multilingual: ZH / JA / EN
- ⚡ 30-step flow matching inference
- 🎵 48kHz high-fidelity output

## 1. Install Dependencies

In [ ]:
# Run this cell once to install dependencies
!pip install -q torch torchaudio transformers diffusers huggingface_hub
!pip install -q phonemizer pykakasi pypinyin rjieba bitsandbytes
!apt-get install -qq espeak-ng > /dev/null 2>&1 || brew install espeak-ng 2>/dev/null || echo 'Please install espeak-ng manually'

## 2. Download Model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

# ============================================================
HF_REPO_ID = "Regeny/VAE-DiT-TTS"

# Download checkpoint and vocab
print("Downloading model files...")
CHECKPOINT_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="checkpoint.pt")
VOCAB_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="char_vocab.json")

# Download inference code
CODE_DIR = snapshot_download(
    repo_id=HF_REPO_ID,
    allow_patterns=["*.py", "*.yaml", "*.json"],
    ignore_patterns=["*.pt", "*.bin", "*.safetensors"],
)

# Download VAE folder
VAE_DIR = snapshot_download(
    repo_id=HF_REPO_ID,
    allow_patterns=["vae/*"],
)

# Add code directory to Python path
import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"Code:       {CODE_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Vocab:      {VOCAB_PATH}")
print("✅ Download complete!")

## 3. Load Models

In [ ]:
import torch
import torchaudio
from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load DiT, TextEncoder, DurationPredictor, FlowMatching
from inference import load_checkpoint
dit, text_encoder, dur_pred, flow, cfg, char_tokenizer = load_checkpoint(
    CHECKPOINT_PATH, device, vocab_path_override=VOCAB_PATH
)
print(f"DiT: {dit.num_params / 1e6:.1f}M params")
print(f"Vocab: {char_tokenizer.vocab_size} chars")

# Load VAE (Oobleck from Stable Audio Open)
from models.vae import load_vae, vae_encode, vae_decode

VAE_PATH = f"{VAE_DIR}/vae"
vae = load_vae(VAE_PATH, device=device, precision="fp16")
print(f"VAE loaded from: {VAE_PATH}")
print(f"Audio: {cfg['audio']['sample_rate']}Hz, {cfg['audio']['latent_rate']}fps latent")
print("✅ All models loaded!")

## 4. Configuration

Set reference audio, text, and inference parameters below.

In [ ]:
# ============================================================
# 🎙️ Reference Audio (for voice cloning)
# ============================================================
PROMPT_AUDIO = "ref_audio.mp3"          # Path to 3-10s reference audio
PROMPT_TEXT  = "誰であれ、戦う心があるのなら──"     # Transcription of reference audio
PROMPT_LANG  = "JA"                  # ZH / JA / EN

# ============================================================
# 📝 Text to Synthesize
# ============================================================
TTS_TEXT = "你好，今天天气真好，我们一起出去玩吧"
TTS_LANG = "ZH"                      # ZH / JA / EN

# ============================================================
# ⚙️ Inference Parameters
# ============================================================
CFG_SCALE = None   # Classifier-free guidance scale (default: 3.0)
N_STEPS   = None   # Sampling steps (default: 30)
DURATION  = None   # Output duration in seconds (None = auto predict)
SEED      = 42     # Random seed (None = random)

## 5. Generate Speech

In [ ]:
from inference import inference

output_path = "output.wav"

gen_latent = inference(
    dit, text_encoder, dur_pred, flow, cfg,
    prompt_audio_path=PROMPT_AUDIO,
    prompt_text=PROMPT_TEXT,
    tts_text=TTS_TEXT,
    prompt_language=PROMPT_LANG,
    tts_language=TTS_LANG,
    char_tokenizer=char_tokenizer,
    vae_encode_fn=lambda wav: vae_encode(vae, wav),
    vae_decode_fn=lambda lat: vae_decode(vae, lat),
    output_path=output_path,
    duration=DURATION,
    cfg_scale=CFG_SCALE,
    n_steps=N_STEPS,
    seed=SEED,
)

print(f"\n✅ Generated: {output_path}")

## 6. Play Audio

In [ ]:
print("🔊 Synthesized Audio")
print(f"   Text: {TTS_TEXT}")
display(Audio(output_path))

print("\n🎙️ Reference Audio")
print(f"   Text: {PROMPT_TEXT}")
display(Audio(PROMPT_AUDIO))

---

## 7. Batch Generation (Optional)

Generate multiple utterances at once.

In [ ]:
test_texts = [
    ("你好，今天天气真好", "ZH"),
    ("我们一起去吃饭吧", "ZH"),
    ("明天你有空吗？我想和你聊聊", "ZH"),
    # ("こんにちは、元気ですか", "JA"),
    # ("Hello, how are you today?", "EN"),
]

for i, (text, lang) in enumerate(test_texts):
    out = f"output_{i}.wav"
    print(f"\n[{i+1}/{len(test_texts)}] {text}")
    inference(
        dit, text_encoder, dur_pred, flow, cfg,
        prompt_audio_path=PROMPT_AUDIO,
        prompt_text=PROMPT_TEXT,
        tts_text=text,
        prompt_language=PROMPT_LANG,
        tts_language=lang,
        char_tokenizer=char_tokenizer,
        vae_encode_fn=lambda wav: vae_encode(vae, wav),
        vae_decode_fn=lambda lat: vae_decode(vae, lat),
        output_path=out,
        seed=SEED,
    )
    display(Audio(out))

## 8. Compare CFG Scales (Optional)

Find the best CFG scale for your use case.

In [ ]:
for cfg_s in [1.0, 2.0, 3.0, 5.0]:
    out = f"output_cfg{cfg_s}.wav"
    print(f"\nCFG scale = {cfg_s}")
    inference(
        dit, text_encoder, dur_pred, flow, cfg,
        prompt_audio_path=PROMPT_AUDIO,
        prompt_text=PROMPT_TEXT,
        tts_text=TTS_TEXT,
        prompt_language=PROMPT_LANG,
        tts_language=TTS_LANG,
        char_tokenizer=char_tokenizer,
        vae_encode_fn=lambda wav: vae_encode(vae, wav),
        vae_decode_fn=lambda lat: vae_decode(vae, lat),
        output_path=out,
        cfg_scale=cfg_s,
        seed=SEED,
    )
    display(Audio(out))